In [32]:
import sksurv #scikit-survival
import pandas as pd
import numpy as np
import sys
import os


sys.path.append(os.path.abspath("../../"))
from src.utils.ConvertTextToCsv import TextToCsv
from src.utils.Preprocessing import Preprocessor
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [33]:
pp = Preprocessor()
df_clinical_data = pd.read_csv("../../data/raw/brca_tcga_pub2015_clinical_data.tsv", sep='\t')
KEEP = [
    
    "Sample ID", "Patient ID",
    "Overall Survival (Months)", "Overall Survival Status",
    "Diagnosis Age",
    "Neoplasm Disease Stage American Joint Committee on Cancer Code",
    "Lymph Node(s) Examined Number",
    "Menopause Status",
    "ER Status By IHC",
    "PR status by ihc",
    "HER2 fish status",
    "HER2 ihc score",
]

df_clinical_keep = df_clinical_data[[c for c in KEEP if c in df_clinical_data.columns]].copy()
    

In [34]:

list_df = pp.total_type_len_type_cancer(df_clinical_keep)
df_clinical_keep["Tumor-Cancer"] = list_df

df_mRNA_transformed = TextToCsv("../../data/raw/data_mrna_seq_v2_rsem.txt")

Luminal A: 330 - Total(%): 0.40
Luminal B: 81 - Total(%):0.10
HER2-enriched: 23 - Total(%):0.03
TNBC: 85 - Total(%)0.10 
UNK: 299 - Total(%) 0.37
Shape of the CSV: (20440, 819)


In [35]:
df_merged = pp.merge_datasets(df_clinical_keep, df_mRNA_transformed)
df_merged

,Sample ID,0,1,2,3,4,5,6,7,8,...,Overall Survival Status,Diagnosis Age,Neoplasm Disease Stage American Joint Committee on Cancer Code,Lymph Node(s) Examined Number,Menopause Status,ER Status By IHC,PR status by ihc,HER2 fish status,HER2 ihc score,Tumor-Cancer
0,TCGA-A1-A0SB-01,14.3935,116.3870,279.7612,0.4505,0.0,0.9010,0.9010,1.8020,0.0000,...,0:LIVING,70.0,Stage I,2.0,Post (prior bilateral ovariectomy OR >12 mo si...,Positive,Negative,NaN,NaN,<UNK>
1,TCGA-A1-A0SD-01,11.3241,60.2630,83.6986,0.3308,0.0,0.6616,0.3308,4.6315,0.3308,...,0:LIVING,59.0,Stage IIA,3.0,NaN,Positive,Positive,NaN,NaN,<UNK>
2,TCGA-A1-A0SE-01,4.4426,153.1452,74.7018,0.0000,0.0,0.0000,0.9872,5.5944,0.3291,...,0:LIVING,56.0,Stage I,8.0,Pre (<6 months since LMP AND no prior bilatera...,Positive,Positive,Negative,1.0,Luminal A
3,TCGA-A1-A0SF-01,10.7401,141.1933,314.4482,0.0000,0.0,0.0000,2.9988,9.4249,0.0000,...,0:LIVING,54.0,Stage IIA,2.0,Pre (<6 months since LMP AND no prior bilatera...,Positive,Positive,NaN,NaN,<UNK>
4,TCGA-A1-A0SH-01,3.0048,79.8003,95.7054,0.0000,0.0,0.0000,0.3612,3.9727,0.0000,...,0:LIVING,39.0,Stage IIA,2.0,Pre (<6 months since LMP AND no prior bilatera...,Negative,Positive,Negative,2.0,<UNK>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
812,TCGA-MS-A51U-01,7.9343,150.6834,540.4278,1.5649,0.0,0.5216,0.0000,6.2598,0.0000,...,0:LIVING,44.0,Stage IIB,18.0,NaN,Positive,Positive,Negative,NaN,Luminal A
813,TCGA-OL-A66H-01,2.2519,115.3378,338.1985,2.0851,0.0,0.0000,2.0851,4.1701,0.0000,...,0:LIVING,NaN,Stage IB,1.0,Post (prior bilateral ovariectomy OR >12 mo si...,Positive,Positive,Negative,NaN,Luminal A
814,TCGA-OL-A66I-01,1.2603,158.3599,210.7460,0.4173,0.0,2.0866,2.5039,2.5039,0.4173,...,0:LIVING,36.0,Stage IIA,13.0,NaN,Negative,Negative,Negative,NaN,TNBC
815,TCGA-OL-A66J-01,5.0428,124.6327,323.1185,0.4507,0.0,0.0000,8.1118,5.8585,3.1546,...,0:LIVING,80.0,Stage I,4.0,Post (prior bilateral ovariectomy OR >12 mo si...,Positive,Positive,Negative,NaN,Luminal A


In [36]:
binary_groups = []
for _, row in df_merged.iterrows():
    ER_status = row.get("ER Status By IHC", pd.NA)
    
    if (ER_status == "Negative"):
        binary_groups.append(0)
    
    elif (ER_status == "Positive"):
        binary_groups.append(1)
    
    else:
        binary_groups.append(-1)

In [37]:
print(len(binary_groups))
print(binary_groups)

df_merged["Binary ER group"] = binary_groups


817
[1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, -1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, -1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, -1, 1, -1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1

In [38]:
binary_groups_df = df_merged.loc[
    df_merged["Binary ER group"].isin([0, 1]),
    ["Binary ER group"] +list(df_merged.columns[1:20441])
]

binary_groups_df["Binary ER group"].unique()

array([1, 0])

In [39]:
binary_reduce_df = pp.elimnation_zeros(binary_groups_df, "Binary ER group")

Max of zeros per row in the dataset: 775
Avg of zeros per row in the dataset: 110.59877690802348
Median of zeros per row in the dataset: 0.0
Min of zeros per row in the dataset: 0
After the 0 elimination: 16264


In [40]:
print(f"Samples: {binary_reduce_df.shape[0]}, Genes: {binary_reduce_df.shape[1]}")

Samples: 775, Genes: 16264


In [41]:
results_df, design, _ = pp.initialize_limma(binary_reduce_df, "Binary ER group")
results_df.rename(columns={"column0" : "Binary 0",
                           "column1": "Binary 1"})

,Binary 0,Binary 1,AveExpr,F,pvalue,adj_pvalue
10190,7.587871,7.643881,7.631233,58628.740884,0.000000e+00,0.000000e+00
10191,8.912371,8.022297,8.223281,20972.390177,0.000000e+00,0.000000e+00
56,9.820627,11.004093,10.736859,46571.716726,0.000000e+00,0.000000e+00
10194,2.864254,3.434416,3.305669,3742.256511,0.000000e+00,0.000000e+00
58,7.734712,8.049891,7.978722,19519.111418,0.000000e+00,0.000000e+00
...,...,...,...,...,...,...
4988,2.404297,2.061032,2.138543,419.623971,5.749432e-183,5.750846e-183
17494,1.954852,2.102671,2.069292,376.615675,2.740884e-164,2.741389e-164
5630,1.656050,1.954234,1.886902,367.969250,1.559500e-160,1.559692e-160
16977,1.783533,3.149988,2.841434,359.632269,6.511637e-157,6.512038e-157
